# 03 - Extract skills from job descriptions

**Goal:** find which skills (SQL, Power BI, Python...) each job asks for, and answer your project questions 2, 3 and 5.

**How it works:** we keep a dictionary of skills, each with a search pattern (for example Power BI can be written `Power BI`, `PowerBI` or `power-bi`).
We scan every job description for these patterns. The result is a "long" table with one row per job-skill pair, which is exactly what SQL and Power BI like.

Run the cells **from top to bottom** with **Shift + Enter**. You need `data/clean/jobs_clean.csv` from the previous notebook.

| Step | What we do |
|---|---|
| 1 | Load the clean data and give each job an ID |
| 2 | Define the skill dictionary |
| 3 | Find skills in every description |
| 4 | Sanity-check the matching |
| 5 | Most demanded skills (question 2) |
| 6 | Skills by role and for freshers (question 5) |
| 7 | Skills that appear together (question 3) |
| 8 | Save the results |

## Step 1 - Load the clean data

In [ ]:
import re
from itertools import combinations
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

jobs = pd.read_csv("../data/clean/jobs_clean.csv")
jobs.insert(0, "job_id", range(1, len(jobs) + 1))   # a unique ID for every job

print("Jobs loaded:", len(jobs))
print("Core-role jobs:", int(jobs["is_core_role"].sum()))
jobs[["job_id", "job_title", "role_group", "experience_level"]].head()

## Step 2 - The skill dictionary
Each skill has a **name**, a **category** and a **pattern**. `\b` in a pattern means "a word boundary", so `sql` does not match inside `nosql`.
You can add or remove skills here later and re-run everything.

Note: the single letter **R** is hard to detect safely (it is also just a letter), so we only count it in clear phrases like "R programming" or "Python/R".

In [ ]:
SKILLS = [
    ("SQL", "Database", r"\bsql\b|\bmysql\b|\bpostgres(?:ql)?\b|\bt-sql\b|\bpl/sql\b"),
    ("Python", "Programming", r"\bpython\b"),
    ("R", "Programming", r"\br programming\b|\br language\b|\brstudio\b|\bin r\b|\bpython\s*(?:/|,|and|or|&)\s*r\b|\br\s*(?:/|,)\s*python\b"),
    ("Pandas/NumPy", "Programming", r"\bpandas\b|\bnumpy\b"),
    ("Excel", "Spreadsheet", r"\bexcel\b|\bvlookup\b|\bpivot tables?\b"),
    ("VBA/Macros", "Spreadsheet", r"\bvba\b|\bmacros?\b"),
    ("Power BI", "BI Tool", r"\bpower\s?-?bi\b|\bpowerbi\b"),
    ("Tableau", "BI Tool", r"\btableau\b"),
    ("Looker/Qlik", "BI Tool", r"\blooker\b|\bqlik(?:view|sense)?\b"),
    ("DAX/Power Query", "BI Tool", r"\bdax\b|\bpower query\b"),
    ("Data Visualization", "Analytics", r"\bvisuali[sz]ation|\bdashboards?\b|\bmatplotlib\b|\bseaborn\b"),
    ("Statistics", "Analytics", r"\bstatistic|\bhypothesis test|\bregression\b|\ba/b test"),
    ("Machine Learning", "Data Science", r"\bmachine learning\b|\bml\b|\bscikit|\bsklearn\b|\bxgboost\b"),
    ("Deep Learning", "Data Science", r"\bdeep learning\b|\btensorflow\b|\bpytorch\b|\bkeras\b"),
    ("NLP/GenAI", "Data Science", r"\bnlp\b|\bnatural language\b|\bllms?\b|\bgenerative ai\b|\bgenai\b"),
    ("ETL/Pipelines", "Data Engineering", r"\betl\b|\belt\b|\bdata pipelines?\b|\bairflow\b"),
    ("Spark/Big Data", "Data Engineering", r"\bspark\b|\bpyspark\b|\bhadoop\b|\bhive\b|\bkafka\b|\bdatabricks\b|\bbig data\b"),
    ("Data Warehousing/Modeling", "Data Engineering", r"\bdata warehous|\bdata model|\bdimensional model"),
    ("Snowflake", "Cloud/Platform", r"\bsnowflake\b"),
    ("AWS", "Cloud/Platform", r"\baws\b|\bamazon web services\b"),
    ("Azure", "Cloud/Platform", r"\bazure\b"),
    ("GCP/BigQuery", "Cloud/Platform", r"\bgcp\b|\bgoogle cloud\b|\bbigquery\b"),
    ("Git/GitHub", "Tools", r"\bgit\b|\bgithub\b"),
    ("SAS/SPSS", "Tools", r"\bsas\b|\bspss\b"),
    ("Communication", "Soft Skill", r"\bcommunication\b|\bstakeholders?\b|\bpresentation skills?\b|\bstorytelling\b"),
    ("Problem Solving", "Soft Skill", r"\bproblem[- ]solving\b|\banalytical (?:skills|thinking|mindset)\b"),
]

compiled = [(name, cat, re.compile(pattern)) for name, cat, pattern in SKILLS]
print(len(compiled), "skills in the dictionary")

## Step 3 - Find the skills in every description
For each job we lower-case the description and test every pattern. A skill counts **once per job**, even if the description mentions it five times.

In [ ]:
rows = []
for job_id, desc in zip(jobs["job_id"], jobs["job_description"]):
    text = str(desc).lower()
    for name, cat, rx in compiled:
        if rx.search(text):
            rows.append((job_id, name, cat))

skills = pd.DataFrame(rows, columns=["job_id", "skill", "category"])
jobs["skill_count"] = jobs["job_id"].map(skills.groupby("job_id").size()).fillna(0).astype(int)

print("Job-skill pairs found:", len(skills))
skills.head(10)

## Step 4 - Sanity-check the matching
Never trust an extraction blindly. We check how many jobs got at least one skill, and read a few examples by eye.

In [ ]:
print("Jobs with at least 1 skill found: {:.1f}%".format((jobs["skill_count"] > 0).mean() * 100))
print()
print("Skills per job:")
print(jobs["skill_count"].describe().round(1))
print()
print("Sample of jobs where NO skill was found (is the description just vague?):")
print(jobs.loc[jobs["skill_count"] == 0, ["job_title", "role_group"]].head(10))

In [ ]:
sample = jobs.sample(3, random_state=1)
for _, row in sample.iterrows():
    found = skills.loc[skills["job_id"] == row["job_id"], "skill"].tolist()
    print("JOB:", row["job_title"])
    print("FOUND:", found)
    print("TEXT:", str(row["job_description"])[:400].replace("\n", " "), "...")
    print("-" * 80)

## Step 5 - Most demanded skills (project question 2)
We measure **the % of core-role jobs that mention each skill**. Core roles are analyst, BI, data scientist, data engineer and ML jobs (not Data Operations or Other).

In [ ]:
core = jobs[jobs["is_core_role"]]
skills_core = skills[skills["job_id"].isin(core["job_id"])]
n_core = len(core)

top_skills = (skills_core.groupby("skill")["job_id"].nunique() / n_core * 100).round(1).sort_values(ascending=False)
print("Core-role jobs:", n_core)
top_skills.head(20)

In [ ]:
top15 = top_skills.head(15).sort_values()
plt.figure(figsize=(8, 6))
plt.barh(top15.index, top15.values)
plt.xlabel("% of core-role jobs mentioning the skill")
plt.title("Most demanded skills")
plt.tight_layout()
plt.show()

## Step 6 - Skills by role, and what a fresher should learn (question 5)
`n` is the number of jobs in each group. **Treat groups with small n (under about 30) as rough hints, not conclusions.**

In [ ]:
def top_skills_for(mask, n=10):
    ids = jobs.loc[mask, "job_id"]
    sub = skills[skills["job_id"].isin(ids)]
    pct = (sub.groupby("skill")["job_id"].nunique() / len(ids) * 100).round(1)
    return len(ids), pct.sort_values(ascending=False).head(n)

groups = {
    "Data Analyst": jobs["role_group"] == "Data Analyst",
    "Data Scientist": jobs["role_group"] == "Data Scientist",
    "Data Engineer": jobs["role_group"] == "Data Engineer",
    "BI + Business Analyst (small sample)": jobs["role_group"].isin(["BI Analyst", "Business Analyst"]),
}

for label, mask in groups.items():
    n, top = top_skills_for(mask)
    print(f"=== {label} (n={n}) ===")
    print(top.to_string())
    print()

In [ ]:
analyst_roles = ["Data Analyst", "BI Analyst", "Business Analyst"]

fresher_mask = jobs["role_group"].isin(analyst_roles) & (jobs["experience_level"] == "Entry/Fresher")
n, top = top_skills_for(fresher_mask, n=12)
print(f"=== FRESHER analyst-type jobs (n={n}) ===")
print(top.to_string())
print()

mid_senior_mask = jobs["role_group"].isin(analyst_roles) & jobs["experience_level"].isin(["Mid (2-4 yrs)", "Senior (5+ yrs)"])
n, top = top_skills_for(mid_senior_mask, n=12)
print(f"=== MID/SENIOR analyst-type jobs (n={n}) ===")
print(top.to_string())

## Step 7 - Skills that appear together (project question 3)
For every job, we list all pairs of skills it asks for, then count the most common pairs.

In [ ]:
pairs = Counter()
for job_id, group in skills_core.groupby("job_id")["skill"]:
    for a, b in combinations(sorted(group), 2):
        pairs[(a, b)] += 1

pair_df = pd.DataFrame([(a, b, c) for (a, b), c in pairs.items()], columns=["skill_1", "skill_2", "jobs"])
pair_df["pct_of_core_jobs"] = (pair_df["jobs"] / n_core * 100).round(1)
pair_df = pair_df.sort_values("jobs", ascending=False).reset_index(drop=True)
pair_df.head(15)

In [ ]:
def job_set(skill):
    return set(skills_core.loc[skills_core["skill"] == skill, "job_id"])

sql, pbi, tab, py, xl = job_set("SQL"), job_set("Power BI"), job_set("Tableau"), job_set("Python"), job_set("Excel")

print("SQL + Power BI  :", len(sql & pbi))
print("SQL + Tableau   :", len(sql & tab))
print("Power BI + Tableau:", len(pbi & tab))
print("SQL + Python    :", len(sql & py))
print("SQL + Excel     :", len(sql & xl))
print("Python + Excel  :", len(py & xl))

## Step 8 - Save the results
Two tidy files for the next steps (SQL and Power BI):
- `jobs_final.csv`: one row per job (without the long description text, to keep it light)
- `job_skills.csv`: one row per job-skill pair

In [ ]:
jobs_final = jobs.drop(columns=["job_description"])
jobs_final.to_csv("../data/clean/jobs_final.csv", index=False, encoding="utf-8")
skills.to_csv("../data/clean/job_skills.csv", index=False, encoding="utf-8")

print("Saved jobs_final.csv:", jobs_final.shape)
print("Saved job_skills.csv:", skills.shape)

## Checkpoint
Send back the printed output of Steps 4, 5, 6 and 7 (the sanity-check numbers, top skills, skills by role, fresher skills, and top pairs).